In [15]:
# filter dataset
!python MT-Preparation/filtering/filter.py ./en-zh.en ./en-zh.zh en zh

Dataframe shape (rows, columns): (231267, 2)
--- Rows with Empty Cells Deleted	--> Rows: 231267
--- Duplicates Deleted			--> Rows: 229646
--- Source-Copied Rows Deleted		--> Rows: 229640
--- Too Long Source/Target Deleted	--> Rows: 224743
--- HTML Removed			--> Rows: 224743
--- Rows will remain true-cased		--> Rows: 224743
--- Rows with Empty Cells Deleted	--> Rows: 224743
--- Source Saved: ./en-zh.en-filtered-wsd.en
--- Target Saved: ./en-zh.zh-filtered-wsd.zh


## Perform BERT-WSD on SoC Computer Cluster

1. **SSH to your SoC Computer Cluster and copy `en-zh.en-filtered-wsd.en` and `en-zh.zh-filtered-wsd.zh` over**  

2. **Run `salloc` to acquire a GPU host:**  
   ```bash
   salloc -G nv

3. **Enter the host `slurm`:** 
    ```bash
    srun --pty bash

4. **Give permission to run script:**
    ```bash
    chmod a+x script.sh

5. **Run the script, feel free to change the nice bonus:**
    ```bash
    ./script.sh

In [ ]:
# script.sh
#!/bin/bash

# Get the number of available processors
NUM_CORES=$(nproc --all)

echo "Detected $NUM_CORES cores. Running wsd.py with batch size = $NUM_CORES..."

# Run the Python script with NUM_CORES as the batch size
nice -n 400 python wsd.py "$NUM_CORES"


In [ ]:
# wsd.py
import sys
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
import pandas as pd
import re
import os

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('wordnet')

class BertWSDProcessor:
    def __init__(self, model_path='bert-base-uncased', device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.tokenizer = BertTokenizer.from_pretrained(model_path)
        self.model = BertForSequenceClassification.from_pretrained(model_path)
        self.model.to(device)
        self.model.eval()
    
    def identify_ambiguous_words(self, sentence):
        """Identify potentially ambiguous words in the sentence"""
        tokens = word_tokenize(sentence)
        ambiguous_words = []
        
        for token in tokens:
            # Check if the word has multiple senses in WordNet
            synsets = wordnet.synsets(token)
            if len(synsets) > 1:
                ambiguous_words.append(token)
                
        return ambiguous_words
    
    def disambiguate_word(self, word, context):
        """Use BERT-WSD to disambiguate a word in context"""
        # Format input for BERT
        inputs = self.tokenizer(context, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        # Get predicted sense ID (implementation depends on your specific BERT-WSD model)
        predicted_sense = outputs.logits.argmax().item()
        
        # Map sense ID to WordNet sense (this mapping depends on your model)
        # For simplicity, we'll just return the sense ID in this example
        return f"{word}#{predicted_sense}"
    
    def process_sentence(self, sentence):
        """Process a sentence, disambiguating ambiguous words"""
        ambiguous_words = self.identify_ambiguous_words(sentence)
        processed_sentence = sentence
        
        for word in ambiguous_words:
            # Get the disambiguated sense
            disambiguated_word = self.disambiguate_word(word, sentence)
            
            # Replace the word with its disambiguated form
            # This simple replacement strategy might need improvement for real use cases
            processed_sentence = re.sub(r'\b' + word + r'\b', disambiguated_word, processed_sentence, 1)
            
        return processed_sentence

def preprocess_file(input_file, output_file, batch_size=32, save_interval=1000):
    """Preprocess an entire file using BERT-WSD with resume support."""
    processor = BertWSDProcessor()

    # Check how many lines are already processed
    processed_lines_count = 0
    if os.path.exists(output_file):
        with open(output_file, 'r', encoding='utf-8') as f:
            processed_lines_count = sum(1 for _ in f)  # Count existing lines
    
    print(f"Resuming from line {processed_lines_count}...")

    # Read input file and skip already processed lines
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()[processed_lines_count:]  # Skip processed lines

    processed_lines = []
    
    # Process remaining lines
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        
        for line in batch:
            processed_line = processor.process_sentence(line.strip())
            processed_lines.append(processed_line)
        
        print(f"Processed {processed_lines_count + min(i+batch_size, len(lines))}/{processed_lines_count + len(lines)} lines")

        # Save every `save_interval` lines
        if len(processed_lines) >= save_interval:
            with open(output_file, 'a', encoding='utf-8') as f:
                f.write('\n'.join(processed_lines) + '\n')
            print(f"Saved {len(processed_lines)} lines to {output_file}")
            processed_lines = []  # Clear the buffer

    # Save any remaining lines
    if processed_lines:
        with open(output_file, 'a', encoding='utf-8') as f:
            f.write('\n'.join(processed_lines) + '\n')
        print(f"Final save: {len(processed_lines)} lines to {output_file}")

    print(f"Preprocessing complete. Output saved to {output_file}")

if __name__ == "__main__":
    batch_size = int(sys.argv[1])
    input_file = "./en-zh.en-filtered-wsd.en"
    output_file = "./en-zh.en-filtered-wsd-processed.en"
    
    preprocess_file(input_file, output_file, batch_size=batch_size)

In [8]:
# train a sentencepiece model for subwording
!python MT-Preparation/subwording/1-train_unigram.py ./en-zh.en-filtered-wsd-processed.en ./en-zh.zh-filtered-wsd.zh

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=./en-zh.en-filtered-wsd-processed.en --model_prefix=source --vocab_size=10000 --hard_vocab_limit=false --split_digits=true --user_defined_symbols=__SEP__
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./en-zh.en-filtered-wsd-processed.en
  input_format: 
  model_prefix: source
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: __SEP__
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_ext

In [18]:
# subword the dataset
!python MT-Preparation/subwording/2-subword.py source.model target.model ./en-zh.en-filtered-wsd.en ./en-zh.zh-filtered-wsd.zh

Source Model: source.model
Target Model: target.model
Source Dataset: ./en-zh.en-filtered-wsd.en
Target Dataset: ./en-zh.zh-filtered-wsd.zh
Done subwording the source file! Output: ./en-zh.en-filtered-wsd.en.subword
Done subwording the target file! Output: ./en-zh.zh-filtered-wsd.zh.subword


In [21]:
# first 3 lines before subwording
!head -n 3 ./en-zh.en-filtered-wsd-processed.en && echo "-----" && head -n 3 ./en-zh.en-filtered-wsd.en && echo "-----" && head -n 3 ./en-zh.zh-filtered-wsd.zh



en
Thank you so#0 much#0, Chris. And it's truly#0 a#0 great#0 honor#0 to have#0 the opportunity to come#0 to this stage#0 twice#0; I#0'm extremely#0 grateful#0.
I#0#0#0 have#0 been#0 blown#0 away#0 by#0 this conference#0, and I want#0 to thank all#0 of you for the many nice#0 comments#0 about#0 what I had#0 to say#0 the other#0 night#0.
-----
en
Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.
I have been blown away by this conference, and I want to thank all of you for the many nice comments about what I had to say the other night.
-----
zh
非常谢谢，克里斯。的确非常荣幸 能有第二次站在这个台上的机会，我真是非常感激。
这个会议真是让我感到惊叹不已，我还要谢谢你们留下的 关于我上次演讲的精彩评论


In [20]:
# first 3 lines after subwording
!head -n 3 ./en-zh.en-filtered-wsd-processed.en.subword && echo "---" && head -n 3 ./en-zh.en-filtered-wsd.en.subword && echo "---" && head -n 3 ./en-zh.zh-filtered-wsd.zh.subword

▁en
▁Thank ▁you ▁so#0 ▁much#0, ▁Chris . ▁And ▁it ' s ▁truly#0 ▁a#0 ▁great#0 ▁honor#0 ▁to ▁have#0 ▁the ▁opportunity ▁to ▁come#0 ▁to ▁this ▁stage#0 ▁twice#0; ▁I#0'm ▁extremely#0 ▁grateful#0.
▁I#0#0#0 ▁have#0 ▁been#0 ▁blown#0 ▁away#0 ▁by#0 ▁this ▁conference#0, ▁and ▁I ▁want#0 ▁to ▁thank ▁all#0 ▁of ▁you ▁for ▁the ▁many ▁nice#0 ▁comment#0 s#0 ▁about#0 ▁what ▁I ▁had#0 ▁to ▁say#0 ▁the ▁other#0 ▁night#0.
---
▁en
▁Thank ▁you ▁so ▁much , ▁Chris . ▁And ▁it ' s ▁truly ▁a ▁great ▁honor ▁to ▁have ▁the ▁opportunity ▁to ▁come ▁to ▁this ▁stage ▁twice ; ▁I ' m ▁extremely ▁grateful .
▁I ▁have ▁been ▁blown ▁away ▁by ▁this ▁conference , ▁and ▁I ▁want ▁to ▁thank ▁all ▁of ▁you ▁for ▁the ▁many ▁nice ▁comment s ▁about ▁what ▁I ▁had ▁to ▁say ▁the ▁other ▁night .
---
▁ z h
▁非常 谢谢 , 克里斯 。 的确 非常 荣幸 ▁能 有 第二次 站在 这个 台上 的机会 , 我 真是 非常 感激 。
▁这个 会议 真是 让我 感到 惊 叹 不 已 , 我 还要 谢谢你们 留下 的 ▁关于 我 上 次 演讲 的 精彩 评论


In [30]:
# split the dataset into training set, development set, and test set
# Development and test sets should be between 100 and 500 segments (here we chose 200)
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 ./en-zh.en-filtered-wsd-processed.en.subword ./en-zh.en-filtered-wsd.en.subword ./en-zh.zh-filtered-wsd.zh.subword

Dataframe shape: (224743, 3)
--- Empty Cells Deleted --> Rows: 224742
--- Wrote Files
Done!
Output files
./en-zh.en-filtered-wsd-processed.en.subword.train
./en-zh.en-filtered-wsd.en.subword.train
./en-zh.zh-filtered-wsd.zh.subword.train
./en-zh.en-filtered-wsd-processed.en.subword.dev
./en-zh.en-filtered-wsd.en.subword.dev
./en-zh.zh-filtered-wsd.zh.subword.dev
./en-zh.en-filtered-wsd-processed.en.subword.test
./en-zh.en-filtered-wsd.en.subword.test
./en-zh.zh-filtered-wsd.zh.subword.test


In [31]:
!wc -l ./*.subword.*

     2000 ./en-zh.en-filtered-wsd-processed.en.subword.dev
     2000 ./en-zh.en-filtered-wsd-processed.en.subword.test
     2000 ./en-zh.en-filtered-wsd-processed.en.subword.test.desubword
   220742 ./en-zh.en-filtered-wsd-processed.en.subword.train
     2000 ./en-zh.en-filtered-wsd.en.subword.dev
     2000 ./en-zh.en-filtered-wsd.en.subword.test
   220742 ./en-zh.en-filtered-wsd.en.subword.train
     2000 ./en-zh.zh-filtered-wsd.zh.subword.dev
     2000 ./en-zh.zh-filtered-wsd.zh.subword.test
     2000 ./en-zh.zh-filtered-wsd.zh.subword.test.desubword
   220742 ./en-zh.zh-filtered-wsd.zh.subword.train
   678226 total


In [32]:
# check the first and last line from each dataset
!echo "---First line---"
!head -n 1 ./*.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 ./*.{train,dev,test}

---First line---
==> ./en-zh.en-filtered-wsd-processed.en.subword.train <==
▁en

==> ./en-zh.en-filtered-wsd.en.subword.train <==
▁en

==> ./en-zh.zh-filtered-wsd.zh.subword.train <==
▁ z h

==> ./en-zh.en-filtered-wsd-processed.en.subword.dev <==
▁So#0 ▁it ' s ▁the ▁combination#0 ▁of ▁these ▁two#0 ▁things#0: ▁it ' s ▁education#0 ▁and ▁the ▁type#0 ▁of ▁neighbors#0 ▁that ▁you ▁have#0, ▁which ▁we ' ll ▁talk#0 ▁about#0 ▁more#0 ▁in#0 ▁a#0 ▁moment#0.

==> ./en-zh.en-filtered-wsd.en.subword.dev <==
▁So ▁it ' s ▁the ▁combination ▁of ▁these ▁two ▁things : ▁it ' s ▁education ▁and ▁the ▁type ▁of ▁neighbors ▁that ▁you ▁have , ▁which ▁we ' ll ▁talk ▁about ▁more ▁in ▁a ▁moment .

==> ./en-zh.zh-filtered-wsd.zh.subword.dev <==
▁所以 这两 样东西 是 联合 起来 的 。 ▁其实 就是 你的 受 教育 程度 和 周围 邻居 的 类型 , ▁我们 一会儿 再 具体 的 谈 一 谈 。

==> ./en-zh.en-filtered-wsd-processed.en.subword.test <==
▁You ▁don ' t ▁believe#0 ▁me ?

==> ./en-zh.en-filtered-wsd.en.subword.test <==
▁You ▁don ' t ▁believe ▁me ?

==> ./en-zh.zh-filtered-wsd.z